# 0.0 Initial Library Import

In [ ]:
# Standard libraries
import sys, math, itertools, warnings, importlib, textwrap, random, ast, re, gc, pickle, json, os, sklearn, xgboost, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import operator
from datetime import datetime, date
from IPython.display import display, HTML
import pyfolio as pf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t, norm, entropy
from scipy.optimize import minimize
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split, TimeSeriesSplit, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb

warnings.filterwarnings('ignore')

# 3.7.16 specific
from typing import List

In [ ]:
# Panda display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None) # Show all content of each column
pd.set_option('display.width', 1000)        # Set the display width to 1000 characters
pd.options.display.float_format = '{:,.5f}'.format
np.set_printoptions(precision=5, suppress=True)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)
print("sklearn:", sklearn.__version__)
print("xgb:", xgboost.__version__)

# 0.1 Data import

## - Function import: formatting

In [ ]:
# My Functions: same directory
user = 'mateo'
model = 'bomo'
model_round = 'round_2'

import _formatting_functions
importlib.reload(_formatting_functions)
from _formatting_functions import color_negative_red, make_pretty, data_format, quarter_hr, event_import, nan_inf_summary

import _optimization_functions_37_v2
importlib.reload(_optimization_functions_37_v2)
from _optimization_functions_37_v2 import (optimization_ranges, process_optmz_minmax, optmz_loop_wrap, process_optmz_breaks, optmz_search_with_exclusions, sum_combinations, create_combinations, text_to_dict, outlier_bound, optmz_loop_wrap_with_exclusions)

import _symulation_functions
importlib.reload(_symulation_functions)
from _symulation_functions import build_meta_sizing_map


In [ ]:
# Main dictionary for time-serie data aggregation
sum_cols = ['mtm_pl', 'entry_pl', 'matched_shares', 'entry_side', 'entry_fees', 'exit_fees', 'exit_shares', 'pl_g', 'pl_n', 'fees']
sum_dict = {key: 'sum' for key in (sum_cols)}

In [ ]:
loc = rf"C:/Users/{user}/OneDrive/Documents/algo-0001/xls/output/{model}/{model_round}/data/"
outdir = Path(loc)

names = [
    # Data with QQQs and XGB prob
    "partition_ins_80_001",
    "partition_ins_20_001",
    "partition_oos_001",

    # Data with multiple symbols and XGB prob
    # "partition_ins_80_001_prob_mult",
    # "partition_ins_20_001_prob_mult",
    # "partition_oos_001_prob_mult",

    # Data with metalabels
    # "partition_ins_80_002_prob_meta",
    # "partition_ins_20_002_prob_meta",
    # "partition_oos_002_prob_meta",

    # Data with all probs (meta, multi, qqq, and lgb)
    "partition_ins_80_002_prob_lgb",
    "partition_ins_20_002_prob_lgb",
    "partition_oos_002_prob_lgb",
]

# Option A: load into a dict
loaded = {
    name: pd.read_parquet(outdir / f"{name}.parquet", engine="pyarrow")
    for name in names
}
partition_ins_80_001 = loaded["partition_ins_80_001"]
partition_ins_20_001 = loaded["partition_ins_20_001"]
partition_oos_001    = loaded["partition_oos_001"]

# partition_ins_80_001_prob_mult = loaded["partition_ins_80_001_prob_mult"]
# partition_ins_20_001_prob_mult = loaded["partition_ins_20_001_prob_mult"]
# partition_oos_001_prob_mult    = loaded["partition_oos_001_prob_mult"]

partition_ins_80_001_prob_lgb = loaded["partition_ins_80_002_prob_lgb"]
partition_ins_20_001_prob_lgb = loaded["partition_ins_20_002_prob_lgb"]
partition_oos_001_prob_lgb    = loaded["partition_oos_002_prob_lgb"]

print({k: v.shape for k, v in loaded.items()})

# For reference from v00
#                Name    obs ncols
# 0  partition_ins_80_001  7273   388
# 1  partition_ins_20_001  1823   388
# 2     partition_oos_001  2279   388
# 7  partition_ins_20  14137   387
# 8  partition_ins_80  56475   387

In [ ]:
partition_ins_80_001_prob_lgb.head(2)

# 0.2 Data merge: Bringing XGB prob trained using multiple names (information rich)

In [ ]:
# Stacking MULTI data for faster merge: this, although quick, implies I am merging some train probs into a test data
#... this happens because I am spliting INS 80/20 by % count not by date, so if the pull in MULTI has more rows, the count skews the date ranges

# Stacking metalabels

df_stack2 = pd.concat(
    {
        "ins_80_001": partition_ins_80_001_prob_lgb,
        "ins_20_001": partition_ins_20_001_prob_lgb,
        "oos_001": partition_oos_001_prob_lgb,
    },
    names=["source"],
).reset_index(level="source")



# 0.3 Data merge: Bringing XGB prob trained for Metalabeling

In [ ]:
keys = ["symbol", "entry_time_s"]

# 1) ensure datetime + truncate to second
partition_ins_80_001 = partition_ins_80_001.copy().drop(columns=["yhat_train_1"])
partition_ins_20_001 = partition_ins_20_001.copy().drop(columns=["yhat_train_1"])
partition_oos_001 = partition_oos_001.copy().drop(columns=["yhat_train_1"])
df_stack2 = df_stack2.copy()

partition_ins_80_001["entry_time_s"] = pd.to_datetime(partition_ins_80_001["entry_time"]).dt.floor("s")
partition_ins_20_001["entry_time_s"] = pd.to_datetime(partition_ins_20_001["entry_time"]).dt.floor("s")
partition_oos_001["entry_time_s"] = pd.to_datetime(partition_oos_001["entry_time"]).dt.floor("s")
df_stack2["entry_time_s"] = pd.to_datetime(df_stack2["entry_time"]).dt.floor("s")

# 2) rename RHS prediction col + keep minimal columns
rhs = (
    df_stack2
    # .rename(columns={"yhat_train_1": "yhat_train_meta_1"})
    .loc[:, ["symbol", "entry_time_s", "yhat_train_1", "yhat_train_mult_1", "yhat_train_meta_1", "yhat_train_lgb_1"]]
)

# 3A) left merge on 80/Train
partition_ins_80_001_merged = partition_ins_80_001.merge(
    rhs,
    on=keys,
    how="left",
    validate="m:1",  # change if needed
)

# 3B) left merge on 20/Test
partition_ins_20_001_merged = partition_ins_20_001.merge(
    rhs,
    on=keys,
    how="left",
    validate="m:1",  # change if needed
)

# 3C) left merge on OOS
partition_oos_001_merged = partition_oos_001.merge(
    rhs,
    on=keys,
    how="left",
    validate="m:1",  # change if needed
)


In [ ]:
# Data with two probabilities: 
# ....(1) called yhat_train_1 and comes from QQQ only training (less data)
# ....(2) called yhat_train_mult_1 and comes from MULTI run only training (more data)... because it was trained using AMZN, AAPL, NFLX, MSFT, ORCL, GOOG, TSLA, and QQQ 

partition_ins_80_002 = partition_ins_80_001_merged.copy().drop(columns=["entry_time_s"])
partition_ins_20_002 = partition_ins_20_001_merged.copy().drop(columns=["entry_time_s"])
partition_oos_002 = partition_oos_001_merged.copy().drop(columns=["entry_time_s"])

In [ ]:
partition_ins_80_002.head(10)

In [ ]:
summary_df = nan_inf_summary(partition_ins_80_001_merged, df_name="partition_ins_80_001_merged")
print("")
summary_df = nan_inf_summary(partition_ins_20_001_merged, df_name="partition_ins_20_001_merged")
print("")
summary_df = nan_inf_summary(partition_oos_001_merged, df_name="partition_oos_001_merged")

#### - Saving optimized data (with all features) and all ML probs (QQQ, MULTI, META and lightGBM)

In [ ]:
loc = rf"C:/Users/{user}/OneDrive/Documents/algo-0001/xls/output/{model}/{model_round}/data/"
outdir = Path(loc)
outdir.mkdir(parents=True, exist_ok=True)

In [ ]:
dfs = {
    "partition_ins_80_002": partition_ins_80_002,
    "partition_ins_20_002": partition_ins_20_002,
    "partition_oos_002":    partition_oos_002,
}

for name, df in dfs.items():
    df.to_parquet(outdir / f"{name}.parquet",
                  engine="pyarrow",   # if missing, use engine="fastparquet"
                  index=False,
                  compression="zstd") # if unavailable, try "snappy"


# 1.0 Non-ML Optimization

### - Importing list of variables to optimize

In [ ]:
vars_path = f'C:/Users/{user}/OneDrive/Documents/algo-0001/xls/input/{model}/_varnames_/'
vars_name = 'data_dictionary_post.xlsx'

df = pd.read_excel(vars_path + vars_name, sheet_name='optimization', usecols=['clean name', 'optimize', 'group', 'subgroup'])
df = df[df['optimize'] == True][['clean name']].reset_index()

# All variables to optimize for single optimization and best break review
optmz_list_all = df['clean name'].tolist()
print(len(optmz_list_all))
print(textwrap.fill(", ".join(optmz_list_all), width = 250))


In [ ]:
probs = ['yhat_train_1', 'yhat_train_mult_1', 'yhat_train_meta_1', 'yhat_train_lgb_1']

optmz_list_all_plus = optmz_list_all + probs

print(f'Optimizable variables: {len(optmz_list_all)}')
print(f'Predicted values added: {len(optmz_list_all_plus)}')

### - Defining optimization ranges and steps

In [ ]:
drop_manual = []
all_ranges, var_stats = optimization_ranges(partition_ins_80_002, optmz_list_all_plus, drop_manual, 20)
all_ranges.drop(all_ranges[all_ranges['steps'] == 0].index, inplace=True)

all_ranges.tail(5)

### - Data Export

In [ ]:
# Data exports for review
xls_outpath = f'C:/Users/{user}/OneDrive/Documents/algo-0001/xls/output/{model}/{model_round}/xls/'
outname = f"optmz_consol_stats {datetime.now().strftime('%Y%m%d')} v1.xlsx"
# var_stats.to_excel(xls_outpath + outname, index = True, engine='openpyxl')
print(xls_outpath + outname)

outname = f"optmz variables 10-90 {datetime.now().strftime('%Y%m%d')} 1.xlsx"
# all_ranges.to_excel(xls_outpath + outname, index = True, engine='openpyxl')

print(xls_outpath + outname)

## 1.1 Optimization of single variables for break discovery

In [ ]:
# Create the dictionary with the index as the key and ['min', 'max'] columns as the values
target_vars = {index: (row[0.1], row[0.9], row['steps']) for index, row in all_ranges.iterrows()}
print(f'Target variables: {len(target_vars)}')

# Pass target dictionary to criteria dictionary that will be used by search algorithm
crit_lst = [{key: (j, '>=')} for key, (element1, element2, element3) in target_vars.items() for j in np.arange(element1, element2, element3)]
print(f'Search criteria: {len(crit_lst)}')

### - Single variables, positive (>=)

In [ ]:
## TESTING ALL INDIVIDUAL TARGETS
# Applying list of criteria - 238635 max capital used in all data
crit_lst = [{key: (j, '>=')} for key, (element1, element2, element3) in target_vars.items() for j in np.arange(element1, element2, element3)]
single_pos0, optmz, optmz_bydate = optmz_loop_wrap(partition_ins_80_002, crit_lst, 210000)


# Post-processing consolidates results (selecting first and rd from bottom as a low bar)
# 'extrm' stands for extreme values (first and nth from last: 2 rows per variable)
# 'all' stands for all values (20 rows per variable)
single_pos_extrm, single_pos_all = process_optmz_minmax(single_pos0, 'Sharpe ratio', -3)
gc.collect()

print(f'63 variables x 20 steps: {len(single_pos_all)}')
print(f'63 variables x 2 extreme values: {len(single_pos_extrm)}')

In [ ]:
dataname = 'bomo_v2_long_ml '
outname = f"- optimization all result - pos {datetime.now().strftime('%Y%m%d')} v1.xlsx"
# single_pos_all.to_excel(xls_outpath + dataname + outname, index = True, engine='openpyxl')
print(xls_outpath + dataname + outname)


### - Lib import

In [ ]:
import _fast_optimization_v1
importlib.reload(_fast_optimization_v1)
from _fast_optimization_v1 import greedy_threshold_search

import _bayesian_optimization_v1
importlib.reload(_bayesian_optimization_v1)
from _bayesian_optimization_v1 import bayesian_greedy_threshold_search

import _dashboard_functions_one_symbol_v2
importlib.reload(_dashboard_functions_one_symbol_v2)
from _dashboard_functions_one_symbol_v2 import dashboard

import _data_explore_functions
importlib.reload(_data_explore_functions)
from _data_explore_functions import cross_tabs, explore_cross, decile_summary, line_chart_grid, create_distance, append_summary, get_summary, reset_summary, count_outliers_by_std

## 1.2 Fast Optimization

### - With all features

In [ ]:
df = partition_ins_80_002

res = greedy_threshold_search(df, optmz_list_all_plus, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("trials_per_feature:", res.logs["trials_per_feature"])


### - Excluding ML-probability

In [ ]:
res = greedy_threshold_search(df, optmz_list_all, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("trials_per_feature:", res.logs["trials_per_feature"])


### - By Branches (ex-ML probability): A, B, C and D

In [ ]:
# (a) Selected, (b) distances, (c) percents, (d) returns 
feats_a = [
    "atr_250", "atr_ema_a", "atr_ema_b", "atr_ktg", "atr_sma_a", "atr_sma_b", "atr_val", "beta", "cmf", "corr_1y", "corr_20d", "d_atr", "d_avol5", "d_avol50", "d_natr", 'beta', 'week_day_cos', 'entry_hr_dec',
    "d_natr_ktg", "d_rsi", "pct_chg_open", "rsi", "rvol", "vol_20d", "vol_5d", "vol_60d", "spy_atr", 'rvol', "spy_rvol", "PCA_Index_full", "entry_hr_dec", "kalmar_q", "fear_greed", 'open_vol_rat', 'vol_acc_rat', 'd_avol5_rat',
        ]

feats_b = [b for b in optmz_list_all if b.startswith("dist_")]
feats_c = [c for c in optmz_list_all if c.startswith("pct_")]
feats_d = [d for d in optmz_list_all if d.startswith("ret_")]
        

In [ ]:
# (a) Selected
res = greedy_threshold_search(df, feats_a, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (b) distances
res = greedy_threshold_search(df, feats_b, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (c) percents
res = greedy_threshold_search(df, feats_c, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")
 
# (d) returns 
res = greedy_threshold_search(df, feats_d, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")


### - By Branches (w. ML probability): A, B, C and D

In [ ]:
# (a) Selected, (b) distances, (c) percents, (d) returns 
feats_a_ml = [
    "atr_250", "atr_ema_a", "atr_ema_b", "atr_ktg", "atr_sma_a", "atr_sma_b", "atr_val", "beta", "cmf", "corr_1y", "corr_20d", "d_atr", "d_avol5", "d_avol50", "d_natr", 'beta', 'week_day_cos', 'entry_hr_dec',
    "d_natr_ktg", "d_rsi", "pct_chg_open", "rsi", "rvol", "vol_20d", "vol_5d", "vol_60d", "spy_atr", 'rvol', "spy_rvol", "PCA_Index_full", "entry_hr_dec", "kalmar_q", "fear_greed", 'open_vol_rat', 'vol_acc_rat', 'd_avol5_rat',
        ] + probs

feats_b_ml = [b for b in optmz_list_all if b.startswith("dist_")] + probs
feats_c_ml = [c for c in optmz_list_all if c.startswith("pct_")] + probs
feats_d_ml = [d for d in optmz_list_all if d.startswith("ret_")] + probs
        

In [ ]:
# (a) Selected
res = greedy_threshold_search(df, feats_a_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (b) distances
res = greedy_threshold_search(df, feats_b_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (c) percents
res = greedy_threshold_search(df, feats_c_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")
 
# (d) returns 
res = greedy_threshold_search(df, feats_d_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")


### - Bayesian, by Branches (ex-ML probability): A, B, C and D

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=feats_a, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_b, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_c, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_d, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])

# for step in res.logs["bo_trace"]:
#     print(step)

### - Bayesian, by Branches (w. ML probability): A, B, C and D

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=feats_a_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_b_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_c_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_d_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])

# for step in res.logs["bo_trace"]:
#     print(step)

## 1.3 Optimization - INS (80/20) and OOS

In [ ]:
crit_lst = [    
        # {'yhat_train_1': (0.3308, '>='), 'pct_ask_lod': (0.0063, '<=')}
        {'pct_ask_lod': (0.0063, '<='), 'dist_ask_arimax2': (0.150, '>=')}
               ]

# INS-80%
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_002, crit_lst, 181390)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, '80% Optmz', 1, 'short', 'max')
print(ins_80.head(5))
print("")

# INS-20%
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_002, crit_lst, 202740)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, '20% Optmz', 1, 'short', 'max')
print(ins_20.head(5))
print("")

# OOS
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, 316195)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, 'OOS Optmz', 1, 'short', 'max')
print(oos.head(20))
print("")


In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots
datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


## 1.4 Alt Optimization - INS (80/20) and OOS

In [ ]:
crit_lst = [    
        # {'fear_greed': (60.0, '>='), 'dist_spy_trade_px_spy_d_ema20': (2.0, '<=')}
        # {'yhat_train_1': (0.28, '>='), 'fear_greed': (60.0, '>='), 'dist_spy_trade_px_spy_d_ema20': (2.0, '<=')},
        # {'yhat_train_1': (0.20, '>='), 'fear_greed': (60.0, '>='), 'dist_spy_trade_px_spy_d_ema20': (2.0, '<='), 'dist_ask_arimax1': (0.00, '>=')},

        # {'yhat_train_mult_1': (0.20, '>='), 'fear_greed': (60.0, '>='), 'dist_spy_trade_px_spy_d_ema20': (2.0, '<=')},
        # {'yhat_train_mult_1': (0.20, '>='), 'fear_greed': (60.0, '>='), 'dist_spy_trade_px_spy_d_ema20': (2.0, '<='), 'dist_ask_arimax1': (0.00, '>=')},
        # {'yhat_train_mult_1': (0.20, '>='), 'fear_greed': (60.0, '>='), 'dist_ask_arimax1': (0.00, '>=')},

        # {'dist_ask_arimax2': (0.150, '>=')},

        # {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_1': (0.25, '>=')},
        # {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_1': (0.25, '>='), 'pct_ask_lod': (0.0063, '<=')}, 
        {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_1': (0.33, '>=')}, 
               ]

# INS-80%
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_002, crit_lst, 181390)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, '80% Optmz', 1, 'short', 'max')
print(ins_80.head(5))
print("")

# INS-20%
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_002, crit_lst, 202740)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, '20% Optmz', 1, 'short', 'max')
print(ins_20.head(5))
print("")

# OOS
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_002, crit_lst, 316195)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, 'OOS Optmz', 1, 'short', 'max')
print(oos.head(20))
print("")


In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots
datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


## 1.5 Optimization w. Metalabel - INS (80/20) and OOS

In [ ]:
crit_lst = [    
      #   {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_1': (0.30, '>='), 'yhat_train_meta_1': (0.24, '>=')}, 
      # {'yhat_train_meta_1': (0.24, '>='), 'dist_ask_arimax2': (0.150, '>=')},
       {'yhat_train_meta_1': (0.24, '>='),'yhat_train_mult_1': (0.20, '>='), 'dist_ask_arimax2': (0.150, '>=')},
               ]

# INS-80%
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_002, crit_lst, 181390)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, '80% Optmz', 1, 'short', 'max')
print(ins_80.head(5))
print("")

# INS-20%
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_002, crit_lst, 202740)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, '20% Optmz', 1, 'short', 'max')
print(ins_20.head(5))
print("")

# OOS
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_002, crit_lst, 316195)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, 'OOS Optmz', 1, 'short', 'max')
print(oos.head(20))
print("")


In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots
datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


## 1.6 Optimization w. LightGBM - INS (80/20) and OOS

In [ ]:
crit_lst = [    
      # {'yhat_train_lgb_1': (0.275, '>=')},
      # {'pct_ask_lod': (0.0063, '<='), 'dist_ask_arimax2': (0.150, '>='), 'yhat_train_lgb_1': (0.275, '>=')}
      # {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_1': (0.33, '>='), 'yhat_train_lgb_1': (0.275, '>=')},
      # {'yhat_train_meta_1': (0.24, '>='), 'dist_ask_arimax2': (0.150, '>='), 'yhat_train_lgb_1': (0.275, '>=')},
      # {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_1': (0.30, '>='), 'yhat_train_meta_1': (0.24, '>='), 'yhat_train_lgb_1': (0.275, '>=')},
      # {'yhat_train_meta_1': (0.24, '>='),'yhat_train_lgb_1': (0.275, '>='), 'dist_ask_arimax2': (0.150, '>=')},  
      {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_lgb_1': (0.275, '>=')},
               ]

# INS-80%
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_002, crit_lst, 181390)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, '80% Optmz', 1, 'short', 'max')
print(ins_80.head(5))
print("")

# INS-20%
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_002, crit_lst, 202740)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, '20% Optmz', 1, 'short', 'max')
print(ins_20.head(5))
print("")

# OOS
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_002, crit_lst, 316195)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, 'OOS Optmz', 1, 'short', 'max')
print(oos.head(20))
print("")


In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots
datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


## 1.7 Other toy runs

In [ ]:
crit_lst = [    
        # {'yhat_train_meta_1': (0.20, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.21, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.22, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.23, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.24, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.25, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.33, '>='), 'dist_ask_arimax2': (0.150, '>=')},

        {'yhat_train_lgb_1': (0.275, '>=')},

               ]

# OOS-20%
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_002, crit_lst, 316195)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, 'OOS Optmz', 1, 'short', 'max')
oos.head(2)


In [ ]:
crit_lst = [    
        # {'yhat_train_meta_1': (0.20, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.21, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.22, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.23, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.24, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.25, '>='), 'dist_ask_arimax2': (0.150, '>=')},
        # {'yhat_train_meta_1': (0.33, '>='), 'dist_ask_arimax2': (0.150, '>=')},

        {'yhat_train_lgb_1': (0.275, '>=')},
               ]

ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_002, crit_lst, 202740)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, '20% Optmz', 1, 'short', 'max')
ins_20.head(2)


## 1.8 Building Meta sizing

In [ ]:
# 1) Build sizing map on training DF (partition_ins_80_002)
df_sized_train, bucket_stats, sizing_map, bucket_edges = build_meta_sizing_map(
    partition_ins_80_002,
    prob_col="yhat_train_meta_1",
    pl_col="pl_g",
    date_col="normed_date",
    time_col="entry_time",
    win_col="wins",
    n_buckets=5,
    multipliers=(0.0, 0.0, 1.0, 1.0, 1.25),
    min_trades_per_bucket=200,
)

print(bucket_stats)

# 2) Apply same frozen map to validation/test DF later:
# df_sized_test = apply_frozen_sizing_map(test_df, "yhat_test_meta_1", bucket_edges, sizing_map)


In [ ]:
def apply_frozen_sizing_map(
    df: pd.DataFrame,
    prob_col: str,
    bucket_edges,
    sizing_map: dict,
    pl_col: str = "pl_g",
    time_col: str = "entry_time",
):
    """
    Apply TRAIN-derived bucket_edges + sizing_map to a new DF (test/live).

    Notes:
    - Uses rank(p) within the provided df to place trades into the same *quantile buckets*.
      This is consistent with how the train buckets were constructed in the earlier code.
    - Produces: meta_bucket, size_mult, sized_pl (if pl_col exists).
    """
    d = df.copy()

    # Parse timestamp robustly (optional, but keeps output ordered)
    if time_col in d.columns:
        ts = pd.to_datetime(d[time_col].astype(str), errors="coerce", utc=True)
        d["ts"] = ts.dt.tz_convert(None)
        d = d.sort_values("ts").reset_index(drop=True)

    # Drop missing prob
    d = d.dropna(subset=[prob_col]).copy()

    # Rank-based quantile assignment (same method as train)
    p = d[prob_col].astype(float)
    r = p.rank(method="first", pct=True)

    d["meta_bucket"] = pd.cut(r, bins=bucket_edges, labels=False, include_lowest=True)
    d["size_mult"] = d["meta_bucket"].map(sizing_map).astype(float)

    # If you have PnL on test, compute sized PnL
    if pl_col in d.columns:
        d["sized_pl"] = d[pl_col].astype(float) * d["size_mult"]

    return d



In [ ]:

# Same as the Train data
sizing_map = {0: 0.0, 1: 0.0, 2: 1.0, 3: 1.0, 4: 1.25}

# Example application to test:
df_sized_test = apply_frozen_sizing_map(
    partition_ins_20_002,          # <-- change to your test df name
    prob_col="yhat_train_meta_1",   # <-- change if different
    bucket_edges=bucket_edges,     # <-- from training build_meta_sizing_map(...)
    sizing_map=sizing_map,
    pl_col="pl_g",
    time_col="entry_time",
)

# Quick summary
print(df_sized_test[["meta_bucket", "size_mult"]].value_counts().sort_index())

if "sized_pl" in df_sized_test.columns:
    print("Raw PnL sum:  ", df_sized_test["pl_g"].sum())
    print("Sized PnL sum: ", df_sized_test["sized_pl"].sum())
    print("Raw avg/trade: ", df_sized_test["pl_g"].mean())
    print("Sized avg/trade:", df_sized_test["sized_pl"].mean())


In [ ]:
def perf_stats(x):
    return pd.Series({
        "count": x.count(),
        "mean": x.mean(),
        "std": x.std(),
        "sharpe_like": x.mean() / x.std() if x.std() > 0 else np.nan,
        "sum": x.sum(),
        "min": x.min(),
        "max": x.max(),
    })

raw_stats = perf_stats(df_sized_test["pl_g"])
sized_stats = perf_stats(df_sized_test["sized_pl"])

print("RAW")
print(raw_stats)
print("\nSIZED")
print(sized_stats)


In [ ]:
print(df_sized_test.groupby("meta_bucket").agg(
    trades=("pl_g", "size"),
    win_rate=("pl_g", lambda x: (x > 0).mean()),
    avg_pl=("pl_g", "mean"),
    avg_sized_pl=("sized_pl", "mean"),
    sum_pl=("pl_g", "sum"),
    sum_sized_pl=("sized_pl", "sum"),
))


In [ ]:
sizing_map = {0: 0.0, 1: 0.0, 2: 1.0, 3: 1.0, 4: 1.25}

# Example application to test:
df_sized_oos = apply_frozen_sizing_map(
    partition_oos_002,          # <-- change to your test df name
    prob_col="yhat_train_meta_1",   # <-- change if different
    bucket_edges=bucket_edges,     # <-- from training build_meta_sizing_map(...)
    sizing_map=sizing_map,
    pl_col="pl_g",
    time_col="entry_time",
)

# Quick summary
print(df_sized_oos[["meta_bucket", "size_mult"]].value_counts().sort_index())

if "sized_pl" in df_sized_oos.columns:
    print("Raw PnL sum:  ", df_sized_oos["pl_g"].sum())
    print("Sized PnL sum: ", df_sized_oos["sized_pl"].sum())
    print("Raw avg/trade: ", df_sized_oos["pl_g"].mean())
    print("Sized avg/trade:", df_sized_oos["sized_pl"].mean())


In [ ]:
raw_stats = perf_stats(df_sized_oos["pl_g"])
sized_stats = perf_stats(df_sized_oos["sized_pl"])

print("RAW")
print(raw_stats)
print("\nSIZED")
print(sized_stats)
print("")

print(df_sized_oos.groupby("meta_bucket").agg(
    trades=("pl_g", "size"),
    win_rate=("pl_g", lambda x: (x > 0).mean()),
    avg_pl=("pl_g", "mean"),
    avg_sized_pl=("sized_pl", "mean"),
    sum_pl=("pl_g", "sum"),
    sum_sized_pl=("sized_pl", "sum"),
))


In [ ]:

dfs = [df_sized_train, df_sized_test, df_sized_oos]

for df in dfs:
    # 1. Update 'pl_n' directly (Multiplication in place)
    # This avoids creating 'sized_pl_n', dropping 'pl_n', and renaming
    df['pl_n'] = df['pl_n'] * df['size_mult']

    # 2. Update 'pl_g' directly
    # Overwrite 'pl_g' with values from 'sized_pl'
    df['pl_g'] = df['sized_pl']
    
    # 3. Drop the now redundant 'sized_pl' column
    df.drop(columns=['sized_pl'], inplace=True)

In [ ]:
crit_lst = [    
      {'dist_ask_arimax2': (0.150, '>='),'yhat_train_mult_1': (0.20, '>='), 'yhat_train_lgb_1': (0.275, '>=')},
               ]

# INS-80%
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(df_sized_train, crit_lst, 181390)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, '80% Optmz', 1, 'short', 'max')
print(ins_80.head(5))
print("")

# INS-20%
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(df_sized_test, crit_lst, 202740)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, '20% Optmz', 1, 'short', 'max')
print(ins_20.head(5))
print("")

# OOS
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(df_sized_oos, crit_lst, 316195)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, 'OOS Optmz', 1, 'short', 'max')
print(oos.head(20))
print("")


In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots
datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


## 2.0 Saving metalabels quantile buckets

In [ ]:
# For KITE production!

def fit_prob_quantile_buckets(train_probs: pd.Series, n_buckets: int = 5):
    p = pd.Series(train_probs).astype(float).dropna().values

    # quantile cutpoints: e.g., 0%, 20%, 40%, 60%, 80%, 100%
    qs = np.linspace(0, 1, n_buckets + 1)
    cuts = np.quantile(p, qs)

    # ensure strictly increasing edges (handle ties)
    eps = 1e-12
    for i in range(1, len(cuts)):
        if cuts[i] <= cuts[i-1]:
            cuts[i] = cuts[i-1] + eps

    # make last edge open-ended
    cuts[0] = -np.inf
    cuts[-1] = np.inf
    return cuts  # length n_buckets+1


def bucketize_prob(p_meta: float, cuts):
    # returns bucket 0..n_buckets-1
    return int(np.searchsorted(cuts, p_meta, side="right") - 1)


def bucketize_series(probs: pd.Series, cuts):
    p = probs.astype(float).values
    b = np.searchsorted(cuts, p, side="right") - 1
    return pd.Series(b, index=probs.index, name="meta_bucket")



In [ ]:
dfx = partition_ins_80_002.copy()

In [ ]:

# --- training ---
cuts = fit_prob_quantile_buckets(dfx["yhat_train_meta_1"], n_buckets=5)

# --- apply to any dataset (test/live) ---
dfx["meta_bucket"] = bucketize_series(dfx["yhat_train_meta_1"], cuts)

# sizing map (example)
sizing_map = {0: 0.0, 1: 0.0, 2: 1.0, 3: 1.0, 4: 1.25}
dfx["size_mult"] = dfx["meta_bucket"].map(sizing_map).astype(float)


In [ ]:
cuts

In [ ]:
dfx.head(2)

In [ ]:
# -----------------------------
# HARD-CODED META BUCKET SETUP
# -----------------------------

META_CUTS = [
    float("-inf"),
    0.00034,   # q20  <-- REPLACE
    0.00095,   # q40  <-- REPLACE
    0.01997,   # q60  <-- REPLACE
    0.92714,   # q80  <-- REPLACE
    float("inf"),
]

# Bucket -> size multiplier (your chosen policy)
META_SIZING_MAP = {
    0: 0.00,
    1: 0.00,
    2: 1.00,
    3: 1.00,
    4: 1.25,
}

def meta_bucket_from_prob(p_meta: float, cuts=META_CUTS) -> int:
    """
    Assign bucket 0..(n_buckets-1) based on fixed cutpoints in probability space.
    Safe for live: handles NaN/None and out-of-range values.
    """
    if p_meta is None:
        return 0
    try:
        p = float(p_meta)
    except Exception:
        return 0
    if not math.isfinite(p):
        return 0

    # np.searchsorted returns insertion index; bucket is index-1
    # side="right" means exact cut value goes to the higher bucket.
    idx = int(np.searchsorted(cuts, p, side="right") - 1)

    # clamp
    n_buckets = len(cuts) - 1
    if idx < 0:
        return 0
    if idx >= n_buckets:
        return n_buckets - 1
    return idx

def size_mult_from_prob(p_meta: float) -> float:
    b = meta_bucket_from_prob(p_meta)
    return float(META_SIZING_MAP.get(b, 0.0))

# Usage
p_meta = yhat_meta_1  # output from your meta model
bucket = meta_bucket_from_prob(p_meta)
size_mult = META_SIZING_MAP.get(bucket, 0.0)

# Then apply to your base shares/risk
shares = int(base_shares * size_mult)


## 3.0 Testing Brooks daily break theory

In [ ]:

# 1. Convert entry_time to datetime objects (handles milliseconds auto-magically)
dfx['entry_time'] = pd.to_datetime(dfx['entry_time'])

# 2. Extract just the Date (removes 09:49:00.882)
dfx['date'] = dfx['entry_time'].dt.date

# 3. Create a Daily DataFrame (One row per Day)
# Since 'hod' and 'lod' are the same for every trade on that specific day, 
# dropping duplicates by date gives us the clean daily stats.
dfx = dfx.sort_values('entry_time')

# 2. Drop duplicates, keeping the LAST occurrence for each date
df_daily = dfx.drop_duplicates(subset=['date'], keep='last').copy()

# 4. Define the logic: Did we break the 11AM levels?
# Break High: Final HOD is higher than the High observed at 11 AM
df_daily['broke_high'] = df_daily['hod'] > df_daily['hod_11am']

# Break Low: Final LOD is lower than the Low observed at 11 AM
df_daily['broke_low'] = df_daily['lod'] < df_daily['lod_11am']

# 5. Check Hypothesis
# Rule: "90% chance HOD or LOD holds".
# Meaning: It is RARE that we break BOTH.
df_daily['rule_failed'] = df_daily['broke_high'] & df_daily['broke_low']

# ---------------------------------------------------------
# STATISTICS
# ---------------------------------------------------------
print(f"Total Trading Days Analyzed: {len(df_daily)}")
print(f"Hypothesis Success Rate: {(~df_daily['rule_failed']).mean():.2%}")
print("-" * 30)
print(f"Days High Held (Range didn't expand up):   {(~df_daily['broke_high']).mean():.2%}")
print(f"Days Low Held (Range didn't expand down):  {(~df_daily['broke_low']).mean():.2%}")
print(f"Days BOTH Broke (Vol expansion/Outside Day): {df_daily['rule_failed'].mean():.2%}")

# ---------------------------------------------------------
# P&L IMPACT (Does this volatility hurt you?)
# ---------------------------------------------------------
# We map the daily outcome back to the individual trades in dfx
dfx['daily_rule_failed'] = dfx['date'].map(df_daily.set_index('date')['rule_failed'])

print("\n--- Average Profit (pl_g) per Trade ---")
print(dfx.groupby('daily_rule_failed')['pl_g'].mean())

In [ ]:
# 1. Calculate Total P&L currently
total_pl_current = dfx['pl_g'].sum()

# 2. Calculate Total P&L if we avoided the "Rule Failed" days
# We filter out trades that happened on days where 'daily_rule_failed' is True
df_optimized = dfx[dfx['daily_rule_failed'] == False]
total_pl_optimized = df_optimized['pl_g'].sum()

print(f"Current Total P&L:   ${total_pl_current:,.2f}")
print(f"Optimized Total P&L: ${total_pl_optimized:,.2f}")
print(f"Improvement:         ${total_pl_optimized - total_pl_current:,.2f}")

In [ ]:
# 1. PREP: Get the FINAL High of the Day for every trade
# We derive this from the 'last' row of the day as established previously
dfx = dfx.sort_values('entry_time')
daily_finals = dfx.drop_duplicates(subset=['date'], keep='last').set_index('date')['hod']
dfx['final_daily_high'] = dfx['date'].map(daily_finals)

# 2. FILTER: Look at ONLY your Winning Trades
winners = dfx[dfx['wins'] == 1].copy()

# 3. CLASSIFY: Did the market break the 11 AM High on these winning days?
# Note: We compare the FINAL high to the 11 AM high.
winners['broke_ceiling'] = winners['final_daily_high'] > winners['hod_11am']

# 4. RESULTS
print(f"Total Winning Trades: {len(winners)}")
print("-" * 40)

# Count Breakdown
break_counts = winners['broke_ceiling'].value_counts(normalize=True)
print(f"Winners where Ceiling BROKE (Trend):   {break_counts.get(True, 0):.2%}")
print(f"Winners where Ceiling HELD  (Range):   {break_counts.get(False, 0):.2%}")

print("-" * 40)
# P&L Breakdown
pl_stats = winners.groupby('broke_ceiling')['pl_g'].agg(['mean', 'median', 'count'])
pl_stats.index = ['Ceiling HELD (Range)', 'Ceiling BROKE (Trend)']
print("Average Profit per Trade:")
print(pl_stats)